# Repulsion Sensitivity: Baseline (R = 3.0) vs Reduced Repulsion (R = 2.5)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASELINE_FILE = "../data/all_runs_merged_full.csv"
NEW_RUN_FILES = [
    "repulsion_2.5/D1_N150.csv",
    "repulsion_2.5/D3_N250.csv",
    "repulsion_2.5/D10_N300.csv",
    "repulsion_2.5/D6_N400.csv",
]

In [2]:
df_baseline = pd.read_csv(BASELINE_FILE)
df_baseline.columns = df_baseline.columns.str.strip().str.strip('"')

base_dir = Path(BASELINE_FILE).parent

new_runs = {}
for f in NEW_RUN_FILES:
    df = pd.read_csv(base_dir / f, skiprows=6, quotechar='"')
    df.columns = df.columns.str.strip().str.strip('"')
    key = f"{int(df['nb-dogs'].iloc[0])}D_{int(df['nb-sheep'].iloc[0])}S_R{df['R-repulsion'].iloc[0]}"
    new_runs[key] = df
    print(f"{f} -> {key} ({len(df)} runs)")

repulsion_2.5/D1_N150.csv -> 1D_150S_R2.5 (100 runs)
repulsion_2.5/D3_N250.csv -> 3D_250S_R2.5 (100 runs)
repulsion_2.5/D10_N300.csv -> 10D_300S_R2.5 (100 runs)
repulsion_2.5/D6_N400.csv -> 6D_400S_R2.5 (100 runs)


In [3]:
def compute_metrics(df, label):
    for col in ["ticks-to-success", "dogs-distance", "mean-spreadness", "lost-sheep"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    done = df["done?"].astype(str).str.strip().str.lower()
    n_ok = (done == "true").sum()
    n = len(df)
    df_ok = df[done == "true"]
    return {
        "Label": label,
        "Runs": n,
        "Success Rate": f"{n_ok}/{n} ({100*n_ok/n:.0f}%)",
        "Mean Ticks": f"{df_ok['ticks-to-success'].mean():.0f}" if len(df_ok) else "\u2014",
        "Std Ticks": f"{df_ok['ticks-to-success'].std():.0f}" if len(df_ok) else "\u2014",
        "Mean Dogs Dist": f"{df_ok['dogs-distance'].mean():.0f}" if len(df_ok) else "\u2014",
        "Mean Spread": f"{df_ok['mean-spreadness'].mean():.1f}" if len(df_ok) else "\u2014",
        "Mean Lost Sheep": f"{df['lost-sheep'].mean():.2f}" if "lost-sheep" in df.columns else "\u2014",
    }

rows = []
for key, df_new in new_runs.items():
    n_sheep = int(df_new["nb-sheep"].iloc[0])
    n_dogs = int(df_new["nb-dogs"].iloc[0])
    r_new = float(df_new["R-repulsion"].iloc[0])

    df_base = df_baseline[(df_baseline["nb-sheep"] == n_sheep) & (df_baseline["nb-dogs"] == n_dogs)]
    r_base = df_base["R-repulsion"].iloc[0] if len(df_base) else "?"

    rows.append(compute_metrics(df_base.copy(), f"{n_dogs}D/{n_sheep}S \u2014 Baseline (R={r_base})"))
    rows.append(compute_metrics(df_new, f"{n_dogs}D/{n_sheep}S \u2014 New (R={r_new})"))

pd.DataFrame(rows).set_index("Label")

,Runs,Success Rate,Mean Ticks,Std Ticks,Mean Dogs Dist,Mean Spread,Mean Lost Sheep
Label,,,,,,,
1D/150S — Baseline (R=3),100,1/100 (1%),8976,nan,8158,40.0,56.85
1D/150S — New (R=2.5),100,67/100 (67%),5434,2414,4941,31.2,12.90
3D/250S — Baseline (R=3),100,5/100 (5%),8912,1093,24877,33.2,16.51
3D/250S — New (R=2.5),100,100/100 (100%),2663,759,6899,22.0,0.00
10D/300S — Baseline (R=3),100,50/100 (50%),5598,1968,49827,19.0,2.24
10D/300S — New (R=2.5),100,93/100 (93%),3569,2195,28886,17.1,0.35
6D/400S — Baseline (R=3),100,0/100 (0%),—,—,—,—,70.42
6D/400S — New (R=2.5),100,92/100 (92%),4527,2306,22909,20.9,0.19
